# 01 Data pipeline

Freeze the FRED DEXUSEU snapshot and build past-only features, forward variance targets, chronological splits, and manifests.

## Setup

In [1]:
# Colab setup: clone the private repo, install deps, mount Drive for artifacts.
# Requires a Colab Secret named GITHUB_TOKEN (fine-grained, Contents read-only on this repo).
import base64, os, subprocess, sys
from google.colab import drive, userdata

REPO = "BerkayPolat-lab/gru-currency-volatility-distillation"
REPO_DIR = "/content/gru-currency-volatility-distillation"

def git(*args):
    # Pass the token as an HTTP header through env vars so it never appears in argv,
    # exception messages, or .git/config, and redact it from any error output.
    token = userdata.get("GITHUB_TOKEN").strip()
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"}
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        stderr = result.stderr.replace(token, "***").replace(basic, "***")
        raise RuntimeError(f"git {args[0]} failed:\n{stderr}") from None

if not os.path.exists(REPO_DIR):
    git("clone", "-q", f"https://github.com/{REPO}.git", REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
sys.path.insert(0, f"{REPO_DIR}/model/src")
os.chdir(f"{REPO_DIR}/model")

drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

Mounted at /content/drive


In [2]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

{'data': {'source': 'FRED',
  'series_id': 'DEXUSEU',
  'start': '2000-01-01',
  'end': '2025-12-31',
  'snapshot_path': 'data/raw/DEXUSEU.csv',
  'interpolate_missing': False},
 'features': {'window': 60,
  'trailing_vol_windows': [5, 20],
  'inputs': ['log_return', 'squared_return', 'vol_5', 'vol_20']},
 'target': {'horizon': 5, 'scaling': 'divide_by_train_mean'},
 'splits': {'train': ['2000-01-01', '2017-12-31'],
  'validation': ['2018-01-01', '2020-12-31'],
  'test': ['2021-01-01', '2025-12-31'],
  'drop_examples_crossing_split': True},
 'models': {'teacher': {'num_layers': 3, 'hidden_size': 128, 'dropout': 0.2},
  'student': {'num_layers': 1, 'hidden_size': 32, 'dropout': 0.0},
  'output_transform': 'softplus'},
 'historical_control': {'window': 20},
 'training': {'optimizer': 'adamw',
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'batch_size': 32,
  'max_epochs': 100,
  'early_stopping_metric': 'val_mae',
  'early_stopping_patience': 10,
  'seeds': [0, 1, 2]},
 'distillat

## Download and freeze the snapshot

Download DEXUSEU for 2000-2025, save a dated CSV, and record its hash in the manifest.

In [3]:
import pandas as pd

exchange_rate = (pd.read_csv(CFG["data"]["snapshot_path"], parse_dates=["observation_date"], index_col="observation_date")["DEXUSEU"].loc[CFG["data"]["start"]:CFG["data"]["end"]].dropna())

print(exchange_rate)

observation_date
2000-01-03    1.0155
2000-01-04    1.0309
2000-01-05    1.0335
2000-01-06    1.0324
2000-01-07    1.0294
               ...  
2025-12-24    1.1780
2025-12-26    1.1766
2025-12-29    1.1756
2025-12-30    1.1765
2025-12-31    1.1736
Name: DEXUSEU, Length: 6518, dtype: float64


## Clean and compute log returns

Keep only observed dates with no interpolation, then compute `r(t) = log(P(t) / P(t-1))`.

In [4]:
import numpy as np

log_returns = np.log(exchange_rate).diff().dropna().rename("log_return")
squared_returns = (log_returns ** 2).rename("squared_return")

In [5]:
print(f"{log_returns.name} max: {log_returns.max():.6f} on {log_returns.idxmax().date()}")
print(f"{log_returns.name} min: {log_returns.min():.6f} on {log_returns.idxmin().date()}")

log_return max: 0.046208 on 2009-03-19
log_return min: -0.030031 on 2008-12-19


In [6]:
rms = np.sqrt((squared_returns).mean())
print(f"{log_returns.name} RMS: {rms:.6f}")

log_return RMS: 0.005795


## Input features

Returns, squared returns, and trailing 5- and 20-observation volatility, all ending at the forecast origin.

In [7]:
vol_5 = np.sqrt(squared_returns.rolling(5).mean()).rename("vol_5")
vol_20 = np.sqrt(squared_returns.rolling(20).mean()).rename("vol_20")

print(vol_5.head())

observation_date
2000-01-04       NaN
2000-01-05       NaN
2000-01-06       NaN
2000-01-07       NaN
2000-01-10    0.0072
Name: vol_5, dtype: float64


In [8]:
print(vol_20.head(20))

observation_date
2000-01-04         NaN
2000-01-05         NaN
2000-01-06         NaN
2000-01-07         NaN
2000-01-10         NaN
2000-01-11         NaN
2000-01-12         NaN
2000-01-13         NaN
2000-01-14         NaN
2000-01-18         NaN
2000-01-19         NaN
2000-01-20         NaN
2000-01-21         NaN
2000-01-24         NaN
2000-01-25         NaN
2000-01-26         NaN
2000-01-27         NaN
2000-01-28         NaN
2000-01-31         NaN
2000-02-01    0.006808
Name: vol_20, dtype: float64


In [9]:
features = pd.concat([log_returns, squared_returns, vol_5, vol_20], axis=1).dropna()
print(features)

                  log_return  squared_return     vol_5    vol_20
observation_date                                                
2000-02-01         -0.002668    7.119880e-06  0.008080  0.006808
2000-02-02          0.003795    1.440256e-05  0.008147  0.005979
2000-02-03          0.012109    1.466285e-04  0.008132  0.006539
2000-02-04         -0.012928    1.671426e-04  0.008197  0.007146
2000-02-07          0.002354    5.540304e-06  0.008256  0.007135
...                      ...             ...       ...       ...
2025-12-24          0.000340    1.153392e-07  0.001685  0.002900
2025-12-26         -0.001189    1.414106e-06  0.001612  0.002818
2025-12-29         -0.000850    7.229555e-07  0.001628  0.002819
2025-12-30          0.000765    5.856437e-07  0.001243  0.002790
2025-12-31         -0.002468    6.090932e-06  0.001336  0.002796

[6498 rows x 4 columns]


## Targets

`v(t,h) = (1/h) * sum r(t+j)^2` for `j = 1..h`, with `h = 5`.

In [22]:
h = CFG["target"]["horizon"]
r2 = features["squared_return"]
rolling_mean = r2.rolling(h).mean()
target = rolling_mean.shift(-h).rename("target")

In [38]:
i = 1000
assert np.isclose(target.iloc[i], r2.iloc[i + 1 : i + 1 + h].mean())

# only the last h rows have no target
assert target.isna().sum() == h and target.tail(h).isna().all()

## Chronological splits

Train 2000-2017, validate 2018-2020, test 2021-2025. Drop examples whose forward target crosses a split boundary.

In [27]:
origin = features.index.to_series()
target_end = origin.shift(-h)

In [28]:
SPLITS = ["train", "validation", "test"]

def assign_split(dates):
    conditions = [dates.between(*CFG["splits"][name]) for name in SPLITS]
    labels = np.select(conditions, SPLITS, default="none")
    return pd.Series(labels, index=dates.index)

In [29]:
split = assign_split(origin)
target_split = assign_split(target_end)

examples = features.assign(
    target=target,
    target_end=target_end,
    split=split,
    target_split=target_split,
)

In [30]:
keep = examples["split"].eq(examples["target_split"])

In [31]:
print(examples.loc[~keep, ["split", "target_split"]].value_counts())
examples = examples[keep]
print(examples["split"].value_counts())

split       target_split
test        none            5
train       validation      5
validation  test            5
Name: count, dtype: int64
split
train         4496
test          1244
validation     743
Name: count, dtype: int64


## Scaling

Fit input scaling on train only. Divide targets by the positive train-period mean without centering.

In [46]:
# print(examples.loc[:, ['target', 'target_end', 'split', 'target_split']])

inputs = CFG["features"]["inputs"]
train_rows = split.eq("train")

In [44]:
input_mean = features.loc[train_rows, inputs].mean()
input_std = features.loc[train_rows, inputs].std()

In [50]:
features_scaled = (features[inputs] - input_mean) / input_std

print(features_scaled)

                  log_return  squared_return     vol_5    vol_20
observation_date                                                
2000-02-01         -0.431824       -0.398983  0.901898  0.420084
2000-02-02          0.596302       -0.309286  0.927526  0.010489
2000-02-03          1.918797        1.319262  0.921687  0.287178
2000-02-04         -2.063884        1.571921  0.946714  0.586687
2000-02-07          0.367037       -0.418437  0.969296  0.581624
...                      ...             ...       ...       ...
2025-12-24          0.046645       -0.485253 -1.538238 -1.509659
2025-12-26         -0.196537       -0.469257 -1.566152 -1.550111
2025-12-29         -0.142629       -0.477770 -1.560088 -1.549583
2025-12-30          0.114354       -0.479461 -1.706870 -1.564276
2025-12-31         -0.399958       -0.411656 -1.671204 -1.561261

[6498 rows x 4 columns]


In [49]:
scale_constant = examples.loc[examples["split"].eq("train"), "target"].mean()

print(scale_constant)

3.9509973270581055e-05


In [53]:
examples = examples.assign(target_scaled=examples["target"] / scale_constant)


print(examples["target_scaled"])

observation_date
2000-02-01    2.016709
2000-02-02    2.083799
2000-02-03    1.465834
2000-02-04    0.636640
2000-02-07    0.823826
                ...   
2025-12-17    0.071844
2025-12-18    0.065740
2025-12-19    0.067043
2025-12-22    0.039097
2025-12-23    0.045199
Name: target_scaled, Length: 6483, dtype: float64
0.009720694401570503 15.64492251178897


In [56]:
print(f"minimum target is: {min(examples["target_scaled"])}.", f"maximum target is: {max(examples["target_scaled"])}.")

minimum target is: 0.009720694401570503. maximum target is: 15.64492251178897.


## Save manifests

Save split, feature, and target manifests plus scaling constants to `artifacts/manifests`.

In [57]:
from numpy.lib.stride_tricks import sliding_window_view

W = CFG["features"]["window"]
values = features_scaled.to_numpy(dtype=np.float32)

In [58]:
windows = sliding_window_view(values, window_shape=W, axis=0).transpose(0, 2, 1)

In [59]:
row = features_scaled.index.get_indexer(examples.index)

In [60]:
has_history = row >= W - 1
examples_w = examples[has_history]
row = row[has_history]

In [61]:
X = windows[row - (W - 1)]
y = examples_w["target_scaled"].to_numpy(dtype=np.float32)

In [62]:
splits = {}
for name in ["train", "validation", "test"]:
    mask = examples_w["split"].eq(name).to_numpy()
    splits[name] = (X[mask], y[mask])

In [63]:
import hashlib, json

snapshot_path = CFG["data"]["snapshot_path"]
with open(snapshot_path, "rb") as f:
    snapshot_sha256 = hashlib.sha256(f.read()).hexdigest()

raw_in_range = prices_raw = pd.read_csv(
    snapshot_path, parse_dates=["observation_date"], index_col="observation_date"
)["DEXUSEU"].loc[CFG["data"]["start"]:CFG["data"]["end"]]

In [64]:
train_w = examples_w["split"].eq("train")
scale_constant = float(examples_w.loc[train_w, "target"].mean())
high_vol_cutoff = float(
    examples_w.loc[train_w, "target"].quantile(CFG["evaluation"]["high_vol_quantile"])
)

In [65]:
def split_summary(name):
    rows = examples_w[examples_w["split"].eq(name)]
    return {
        "first_origin": rows.index[0].date().isoformat(),
        "last_origin": rows.index[-1].date().isoformat(),
        "last_target_day": rows["target_end"].iloc[-1].date().isoformat(),
        "examples": len(rows),
        "high_vol_examples": int((rows["target"] > high_vol_cutoff).sum()),
    }

In [66]:
manifest = {
    "created": pd.Timestamp.today().date().isoformat(),
    "snapshot": {
        "file": snapshot_path,
        "sha256": snapshot_sha256,
        "rows_in_range": len(raw_in_range),
        "rows_missing_dropped": int(raw_in_range.isna().sum()),
        "observed_prices": int(raw_in_range.notna().sum()),
        "first_date": raw_in_range.dropna().index[0].date().isoformat(),
        "last_date": raw_in_range.dropna().index[-1].date().isoformat(),
    },
    "examples": {
        "dropped_insufficient_history": int((~has_history).sum()),
        "dropped_crossing_boundary": int((~keep & target_split.ne("none")).sum()),
        "dropped_no_future_target": int(target_split.eq("none").sum()),
        "splits": {name: split_summary(name) for name in ["train", "validation", "test"]},
    },
    "features": {
        "window": W,
        "inputs": inputs,
        "input_mean": {k: float(v) for k, v in input_mean.items()},
        "input_std": {k: float(v) for k, v in input_std.items()},
    },
    "target": {
        "horizon": h,
        "scale_constant": scale_constant,
        "high_vol_quantile": CFG["evaluation"]["high_vol_quantile"],
        "high_vol_cutoff": high_vol_cutoff,
    },
}

In [67]:
for folder in ["artifacts/manifests", f"{ARTIFACTS}/manifests"]:
    os.makedirs(folder, exist_ok=True)
    with open(f"{folder}/data_manifest.json", "w") as f:
        json.dump(manifest, f, indent=2)